In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_json_path = 'rca100/cases/t006/task.json'
with open(task_json_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task.json
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "frontend异常告警" (frontend exception alert). It occurred between 2026-03-28 15:58:01 and 2026-03-28 16:03:05 (UTC+8). The entity that triggered the alert is an APM operation named "frontend::POST" with the entity ID `c6388887e91aa969bdf210fe96a48098`. The user is requesting root cause analysis for this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('frontend异常告警', {'start': '2026-03-28T15:58:01.328491+08:00', 'end': '2026-03-28T16:03:05+08:00'}, {'entity_id': 'c6388887e91aa969bdf210fe96a48098', 'entity_name': 'frontend::POST', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">frontend异常告警<alert_event event_id="27cd9c76f10e12544b38c71b8710eda9" trans_id="l5nv3hueg91uuaen5d6t5e41n6" rule_id="d4107ac6-2a37-4167-9059-16156c90f2bf" rule_name="frontend异常告警" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

# Load the topology.json file to understand the service dependency graph
topology_json_path = 'rca100/cases/t006/topology.json'
with open(topology_json_path, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges from the topology data
entities = topology_data.get('entities', [])
edges = topology_data.get('edges', [])

# Identify upstream and downstream services of the 'frontend' service
frontend_service_id = next((entity['id'] for entity in entities if entity['name'] == 'frontend' and entity['type'] == 'apm.service'), None)

# Filter edges to find upstream and downstream services
upstream_services = [edge['src'] for edge in edges if edge['dst'] == frontend_service_id and edge['relation'] == 'calls']
downstream_services = [edge['dst'] for edge in edges if edge['src'] == frontend_service_id and edge['relation'] == 'calls']

# Map service IDs to their names
upstream_service_names = [entity['name'] for entity in entities if entity['id'] in upstream_services]
downstream_service_names = [entity['name'] for entity in entities if entity['id'] in downstream_services]

upstream_service_names, downstream_service_names

```
Out[2]:
```
The 'frontend' service has one upstream service, 'frontend-proxy'. Its downstream services include 'recommendation', 'checkout', 'ad', 'cart', and 'product-catalog'. Note that some downstream services appear multiple times in the dependency graph.

The original code execution output of IPython Kernel is also provided below for reference:

(['frontend-proxy'], ['recommendation', 'checkout', 'ad', 'cart', 'product-catalog', 'product-catalog', 'recommendation', 'ad', 'cart', 'checkout'])```
```

In [ ]:
# In[3]:

import pandas as pd

# Load the metrics.parquet file
metrics_path = 'rca100/cases/t006/metrics.parquet'
metrics_df = pd.read_parquet(metrics_path)

# Convert 'time' and 'value' columns to numeric types
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Filter metrics for the relevant services (possible root cause components)
root_cause_components = ['frontend', 'frontend-proxy', 'recommendation', 'checkout', 'ad', 'cart', 'product-catalog']
filtered_metrics = metrics_df[metrics_df['entity_name'].isin(root_cause_components)]

# Calculate global thresholds (e.g., P95) for each 'component-KPI' time series
thresholds = filtered_metrics.groupby(['entity_name', 'metric'])['value'].quantile(0.95).reset_index()
thresholds.rename(columns={'value': 'P95_threshold'}, inplace=True)

thresholds

```
Out[3]:
```
The global P95 thresholds for KPIs across the possible root cause components (services) have been calculated. Key observations include:

- The 'frontend' service has high error counts (P95: 101,646) and workload (P95: 120,663), indicating potential issues.
- 'frontend-proxy' also shows significant error counts (P95: 50,035) and workload (P95: 59,909).
- 'cart' and 'checkout' services exhibit high latency (P95: 4.515 seconds and 0.429 seconds, respectively) and slow request counts.
- 'ad', 'product-catalog', and 'recommendation' services have relatively lower thresholds, suggesting they may not be the primary contributors to the issue.

The 'frontend' and 'frontend-proxy' services stand out as likely root cause components due to their high error rates and workloads.

The original code execution output of IPython Kernel is also provided below for reference:

entity_name                       metric  P95_threshold
0                ad  avg_request_latency_seconds   4.356374e-02
1                ad                          cpu   3.635571e-03
2                ad                      latency   4.356374e-02
3                ad                          mem   3.094696e+08
4                ad                request_count   2.177500e+03
5                ad                     workload   2.177500e+03
6              cart                          cpu   1.780166e-01
7              cart                        error   2.777400e+03
8              cart                      latency   4.515165e+00
9              cart                          mem   1.279013e+08
10             cart                         slow   3.519000e+02
11             cart                     workload   2.315300e+04
12         checkout  avg_request_latency_seconds   4.296553e-01
13         checkout                          cpu   1.477558e-01
14         checkout                      latency   4.296553e-01
15         checkout                          mem   5.145696e+07
16         checkout                request_count   5.746600e+03
17         checkout                         slow   1.503100e+03
18         checkout                   slow_count   1.509600e+03
19         checkout                     workload   5.746600e+03
20         frontend                          cpu   7.643160e-01
21         frontend                        error   1.016468e+05
22         frontend                      latency   1.525700e+00
23         frontend                          mem   3.280359e+08
24         frontend                         slow   4.436600e+03
25         frontend                     workload   1.206633e+05
26   frontend-proxy                          cpu   1.089249e-01
27   frontend-proxy                        error   5.003540e+04
28   frontend-proxy                      latency   5.454018e-01
29   frontend-proxy                          mem   5.768602e+07
30   frontend-proxy                         slow   2.435300e+03
31   frontend-proxy                     workload   5.990980e+04
32  product-catalog                          cpu   2.799830e-02
33  product-catalog                      latency   7.568156e-05
34  product-catalog                          mem   3.996522e+07
35  product-catalog                     workload   2.045750e+04
36   recommendation                          cpu   1.396072e-01
37   recommendation                      latency   3.274346e-02
38   recommendation                          mem   2.323750e+08
39   recommendation                         slow   8.000000e+00
40   recommendation                     workload   1.189600e+03```
```

In [ ]:
# In[4]:

import pytz
from datetime import datetime

# Define the time window in UTC+8
timezone = pytz.timezone('Asia/Shanghai')
start_time = int(datetime.strptime('2026-03-28T15:58:01.328491+08:00', '%Y-%m-%dT%H:%M:%S.%f%z').timestamp() * 1_000_000)
end_time = int(datetime.strptime('2026-03-28T16:03:05+08:00', '%Y-%m-%dT%H:%M:%S%z').timestamp() * 1_000_000)

# Filter metrics data for the time window and relevant components
filtered_time_window_metrics = filtered_metrics[
    (filtered_metrics['time'] >= start_time) & (filtered_metrics['time'] <= end_time)
]

# Merge with thresholds to compare KPI values against P95 thresholds
anomalies = filtered_time_window_metrics.merge(
    thresholds, on=['entity_name', 'metric'], how='left'
)

# Detect anomalies where the value exceeds the P95 threshold
anomalies['is_anomalous'] = anomalies['value'] > anomalies['P95_threshold']

# Focus on consecutive anomalies to identify faults
consecutive_anomalies = anomalies[anomalies['is_anomalous']].sort_values(by=['entity_name', 'metric', 'time'])

consecutive_anomalies

```
Out[4]:
```
During the alert time window (2026-03-28 15:58:01 to 16:03:05 UTC+8), anomalies were detected in the following components:

1. **Ad service**:
   - Memory usage exceeded the P95 threshold (309.5 MB) at two timestamps.
   - Workload exceeded the P95 threshold (2,177.5 requests) at two timestamps.

2. **Recommendation service**:
   - Memory usage exceeded the P95 threshold (232.4 MB) at one timestamp.
   - Workload exceeded the P95 threshold (1,189.6 requests) at one timestamp.

These anomalies indicate potential issues in the 'ad' and 'recommendation' services during the incident time window.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set                         entity_id     entity_name  ...         value                           metric_set_id service P95_threshold  is_anomalous
296  1774684921000000    apm  apm.service.legacy                                                ad  ...  3.094897e+08                                    None    None  3.094696e+08          True
368  1774684981000000    apm  apm.service.legacy                                                ad  ...  3.096044e+08                                    None    None  3.094696e+08          True
4    1774684771000000    apm        apm.instance  a3b861321047958ff0ef01d09e9530f6              ad  ...  2.297000e+03  apm@metric_set@apm.metric.apm.instance    None  2.177500e+03          True
117  1774684771000000    apm  apm.service.legacy                                                ad  ...  2.297000e+03                                    None    None  2.177500e+03          True
291  1774684891000000    apm  apm.service.legacy                                    recommendation  ...  2.326801e+08                                    None    None  2.323750e+08          True
149  1774684771000000    apm  apm.service.legacy                                    recommendation  ...  1.218000e+03                                    None    None  1.189600e+03          True

[6 rows x 11 columns]```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_path = 'rca100/cases/t006/traces.parquet'
traces_df = pd.read_parquet(traces_path)

# Convert 'startTime' and 'endTime' to numeric types
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')

# Filter traces within the alert time window
filtered_traces = traces_df[
    (traces_df['startTime'] >= start_time * 1_000) & (traces_df['endTime'] <= end_time * 1_000)
]

# Filter traces involving 'frontend' and its downstream services
relevant_services = ['frontend', 'ad', 'recommendation', 'checkout', 'cart', 'product-catalog']
filtered_traces = filtered_traces[filtered_traces['serviceName'].isin(relevant_services)]

# Identify faulty traces (non-zero statusCode or error in statusMessage)
filtered_traces['statusCode'] = pd.to_numeric(filtered_traces['statusCode'], errors='coerce')
faulty_traces = filtered_traces[
    (filtered_traces['statusCode'] != 0) | (filtered_traces['statusMessage'].notna())
]

# Sort faulty traces by duration to identify the most downstream faulty service
faulty_traces_sorted = faulty_traces.sort_values(by=['duration'], ascending=False)

faulty_traces_sorted

```
Out[5]:
```
The trace analysis during the alert time window identified multiple faulty traces involving the 'frontend' service and its downstream services. Notably, the 'recommendation' and 'ad' services appear frequently in the faulty traces, with operations such as `GET /api/recommendations` and `grpc.oteldemo.AdService/GetAds` showing errors or issues. These services are likely the most downstream faulty components in the call chain and should be investigated further.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                                           spanName  ...  __topic__      __tag__:__pack_id__                                             events           ip links
414484  fac8e747ca7f9cd96c84b04bb8a40836  5e00495b18020502  3489cfb3c74618c1    2                           GET /api/recommendations  ...             6496CEE1E3088D5B-23D616                                               None         None  None
428240  4c39fa33fc5c40d829d5768528db4d64  bee4730290b494c2  70d7bdbf4f7a75a7    3                     grpc.oteldemo.AdService/GetAds  ...             6496CEE1E3088D5B-23D626                                               None         None  None
503886  8507c351914e4a68a20a622c56101c76  a82e6e0e1c76128b  730bf018d99bfd8d    2                                                GET  ...             D208583D15FFC54A-24192E                                               None         None  None
459285  9ef73ebafafcd13e25e89e91458949ce  4b3125da7bc83e01  967be82f303f3230    1   executing api route (pages) /api/recommendations  ...             B6383969769DA797-23FA24                                               None         None  None
468421  de6ce58acb592ebf6b0df3c01bc79629  f91042d5a07e46bd  945430d807dbc8db    1  executing api route (pages) /api/products/[pro...  ...             22D2A3D145F2EFD3-23F5BA                                               None         None  None
...                                  ...               ...               ...  ...                                                ...  ...        ...                      ...                                                ...          ...   ...
477377  0eebe814b30789f6bf6820def1a76d88  8e0f9d5b65c132b7  93bc7d20b03b64ba    3                     grpc.oteldemo.AdService/GetAds  ...             22D2A3D145F2EFD3-23F5C7                                               None         None  None
450557  0fec2c0ef9f7f3959b6713cbc56e61b7  894d67f5b02ca536  7083292a4827b7cf    2                                                GET  ...             B6383969769DA797-23FA0F                                               None         None  None
467016  54f142abedca7be025583e8da32b451d  7bdc82d51a905681  8154cf81cbf73a0f    1                                   get_product_list  ...             E0927E0017BE83F2-24158F                                               None  10.0.16.160  None
487864  cf886c36359532537d04c08b79c74798  67ef0bd288831a28  9d6b8982cd90add5    2          oteldemo.ProductCatalogService/GetProduct  ...             BD9FC65C7D0A6F41-2439DE  [{"attributes":{},"name":"Product Found","time...         None  None
477881  9d12bd30323f9440dbe56ffe412ebefe  210d09566a0b6325  e9bc0216ce56813a    2  /oteldemo.RecommendationService/ListRecommenda...  ...             E0927E0017BE83F2-2415A1                                               None  10.0.16.160  None

[76143 rows x 21 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_path = 'rca100/cases/t006/logs.parquet'
logs_df = pd.read_parquet(logs_path)

# Convert '_time_' to numeric if necessary and filter logs within the alert time window
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')
filtered_logs = logs_df[
    (logs_df['_time_'] >= start_time) & (logs_df['_time_'] <= end_time)
]

# Filter logs related to 'recommendation' and 'ad' services
relevant_logs = filtered_logs[
    filtered_logs['_container_name_'].str.contains('recommendation|ad', case=False, na=False)
]

# Search for error messages or anomalies in the logs
error_logs = relevant_logs[relevant_logs['content'].str.contains('error|exception|fail|critical', case=False, na=False)]

error_logs

```
Out[6]:
```
No error messages or anomalies were found in the logs for the 'recommendation' and 'ad' services during the alert time window. This suggests that the root cause may not be directly observable in the application logs for these services. Further investigation into other telemetry data or upstream dependencies may be needed.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, _image_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```